## Dataset Cleaning and Preprocessing

In [7]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Load dataset
df = pd.read_csv(r"Dataset/Churn_Modelling.csv")

# Remove identifier columns that do not help prediction
df = df.drop(columns=["RowNumber", "CustomerId", "Surname"])

# Basic cleanup
initial_rows = len(df)
duplicate_rows = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)

# Separate target and features
X = df.drop(columns=["Exited"])
y = df["Exited"]

# Detect feature types
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "string"]).columns.tolist()

# Preprocessing pipelines
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"Original rows: {initial_rows}")
print(f"Duplicate rows removed: {duplicate_rows}")
print(f"Missing values after cleanup: {int(df.isna().sum().sum())}")
print(f"Numeric features: {numeric_features}")
print(f"Categorical features: {categorical_features}")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

Original rows: 10000
Duplicate rows removed: 0
Missing values after cleanup: 0
Numeric features: ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']
Categorical features: ['Geography', 'Gender']
X_train shape: (8000, 10)
X_test shape: (2000, 10)


## Encoding Categorical Features

In [8]:
# Cell: Encode categorical features and produce DataFrames
# Fits the existing `preprocessor` on X_train and transforms both splits
import pandas as pd
import numpy as np

# Fit and transform (preprocessor and splits are defined in previous cell)
preprocessor.fit(X_train)
X_train_enc = preprocessor.transform(X_train)
X_test_enc = preprocessor.transform(X_test)

# Obtain feature names
try:
    feature_names = preprocessor.get_feature_names_out()
except Exception:
    num_names = numeric_features
    onehot = preprocessor.named_transformers_["cat"].named_steps["onehot"]
    cat_names = onehot.get_feature_names_out(categorical_features)
    feature_names = np.array(list(num_names) + list(cat_names))

# Convert to DataFrame (ensure dense)
if hasattr(X_train_enc, "toarray"):
    X_train_enc = X_train_enc.toarray()
    X_test_enc = X_test_enc.toarray()

X_train_enc = pd.DataFrame(X_train_enc, columns=feature_names, index=X_train.index)
X_test_enc = pd.DataFrame(X_test_enc, columns=feature_names, index=X_test.index)

# Quick verification
print("Encoded feature count:", X_train_enc.shape[1])
print("X_train_enc shape:", X_train_enc.shape)
print("X_test_enc shape:", X_test_enc.shape)
print("Sample encoded columns:", list(X_train_enc.columns[:10]))

Encoded feature count: 13
X_train_enc shape: (8000, 13)
X_test_enc shape: (2000, 13)
Sample encoded columns: ['num__CreditScore', 'num__Age', 'num__Tenure', 'num__Balance', 'num__NumOfProducts', 'num__HasCrCard', 'num__IsActiveMember', 'num__EstimatedSalary', 'cat__Geography_France', 'cat__Geography_Germany']


## Training the Classification Model

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Train Logistic Regression model on encoded features
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_enc, y_train)

# Predict on test set
y_pred = log_reg.predict(X_test_enc)

# Evaluate
accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Logistic Regression Accuracy: {accuracy:.4f}")
print("Confusion Matrix:\n", cm)
print("\nClassification Report:\n", report)

Logistic Regression Accuracy: 0.8080
Confusion Matrix:
 [[1540   53]
 [ 331   76]]

Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.97      0.89      1593
           1       0.59      0.19      0.28       407

    accuracy                           0.81      2000
   macro avg       0.71      0.58      0.59      2000
weighted avg       0.78      0.81      0.77      2000



## Feature Importance Analysis

In [17]:
from sklearn.linear_model import LogisticRegression
import pandas as pd

# Train a fresh Logistic Regression model for coefficient analysis
importance_model = LogisticRegression(max_iter=1000, random_state=42)
importance_model.fit(X_train_enc, y_train)

# Build encoded coefficient table using absolute values for importance ranking
encoded_coefficients = pd.DataFrame({
    "feature": X_train_enc.columns,
    "coefficient": importance_model.coef_[0],
    "abs_coefficient": abs(importance_model.coef_[0]),
}).sort_values("abs_coefficient", ascending=False)

# Aggregate one-hot encoded columns back to original feature names

def get_original_feature(name: str) -> str:
    base_name = name.split("__", 1)[-1]
    if base_name in numeric_features:
        return base_name
    for feature in categorical_features:
        if base_name == feature or base_name.startswith(f"{feature}_"):
            return feature
    return base_name.split("_", 1)[0]

aggregated_importance = (
    encoded_coefficients.assign(original_feature=encoded_coefficients["feature"].map(get_original_feature))
    .groupby("original_feature", as_index=False)["abs_coefficient"]
    .sum()
    .sort_values("abs_coefficient", ascending=False)
    .reset_index(drop=True)
)

print("Top encoded features by absolute coefficient:")
print(encoded_coefficients.head(10).to_string(index=False))
print("\nAggregated feature importance (original features):")
print(aggregated_importance.to_string(index=False))
print("\nTop churn drivers:")
print(aggregated_importance.head(5).to_string(index=False))

Top encoded features by absolute coefficient:
               feature  coefficient  abs_coefficient
              num__Age     0.739223         0.739223
      cat__Gender_Male    -0.682052         0.682052
 cat__Geography_France    -0.569897         0.569897
  cat__Geography_Spain    -0.523565         0.523565
   num__IsActiveMember    -0.515791         0.515791
cat__Geography_Germany     0.253484         0.253484
          num__Balance     0.160909         0.160909
    cat__Gender_Female    -0.157927         0.157927
      num__CreditScore    -0.086010         0.086010
    num__NumOfProducts    -0.070181         0.070181

Aggregated feature importance (original features):
original_feature  abs_coefficient
       Geography         1.346946
          Gender         0.839979
             Age         0.739223
  IsActiveMember         0.515791
         Balance         0.160909
     CreditScore         0.086010
   NumOfProducts         0.070181
 EstimatedSalary         0.047776
       HasCrC